# Training Analysis
This notebook evaluates the trained YOLO detector on the test set.
 IoU distribution, class accuracy, visual detections, and failure cases.

In [ ]:
import os
import sys
sys.path.append('../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import random
from pathlib import Path
from data.preprocess import make_tf_records_detection_datasets
from training.loss import yolo_loss, iou_metric, class_accuracy

## Load Model and Test Dataset

In [ ]:
model = tf.keras.models.load_model(
    '../models/detector_v13_iou.keras',
    custom_objects={'yolo_loss': yolo_loss, 'iou_metric': iou_metric, 'class_accuracy': class_accuracy}
)

train_dataset, val_dataset, test_dataset, train_size, val_size, test_size = \
    make_tf_records_detection_datasets('../data/custom/detection.tfrecord', 32, 0.7)

print(f'Test set size: {test_size} images')

## IoU Distribution
Compute per-cell IoU between predicted and ground truth boxes across the test set, restricted to cells that contain a hand.

In [ ]:
def compute_iou_per_cell(y_true, y_pred):
    true_boxes = y_true[..., 1:5]
    pred_boxes = y_pred[..., 1:5]
    objectness_mask = y_true[..., 0]

    true_x1 = true_boxes[..., 0] - true_boxes[..., 2] / 2
    true_x2 = true_boxes[..., 0] + true_boxes[..., 2] / 2
    true_y1 = true_boxes[..., 1] - true_boxes[..., 3] / 2
    true_y2 = true_boxes[..., 1] + true_boxes[..., 3] / 2

    pred_x1 = pred_boxes[..., 0] - pred_boxes[..., 2] / 2
    pred_x2 = pred_boxes[..., 0] + pred_boxes[..., 2] / 2
    pred_y1 = pred_boxes[..., 1] - pred_boxes[..., 3] / 2
    pred_y2 = pred_boxes[..., 1] + pred_boxes[..., 3] / 2

    inter_x1 = np.maximum(true_x1, pred_x1)
    inter_x2 = np.minimum(true_x2, pred_x2)
    inter_y1 = np.maximum(true_y1, pred_y1)
    inter_y2 = np.minimum(true_y2, pred_y2)

    inter = np.maximum(0, inter_x2 - inter_x1) * np.maximum(0, inter_y2 - inter_y1)
    true_area = (true_x2 - true_x1) * (true_y2 - true_y1)
    pred_area = (pred_x2 - pred_x1) * (pred_y2 - pred_y1)
    union = true_area + pred_area - inter

    iou = inter / (union + 1e-7)
    return iou[objectness_mask == 1].flatten()

all_ious = []
for imgs, labels in test_dataset.take(test_size // 32 + 1):
    preds = model(imgs, training=False).numpy()
    ious = compute_iou_per_cell(labels.numpy(), preds)
    all_ious.extend(ious)

all_ious = np.array(all_ious)
print(f'Mean IoU:   {all_ious.mean():.3f}')
print(f'Median IoU: {np.median(all_ious):.3f}')
print(f'IoU > 0.5:  {(all_ious > 0.5).mean():.1%} of positive cells')

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(all_ious, bins=30, color='steelblue', edgecolor='white')
plt.axvline(all_ious.mean(), color='red', linestyle='--', label=f'Mean = {all_ious.mean():.2f}')
plt.title('IoU Distribution on Test Set (Positive Cells Only)')
plt.xlabel('IoU')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

## Classification Accuracy on Detected Hands
For cells where the model correctly detects a hand (objectness > threshold), how often does it predict the correct digit class?

In [ ]:
correct = 0
total = 0
class_correct = [0] * 6
class_total = [0] * 6

for imgs, labels in test_dataset.take(test_size // 32 + 1):
    preds = model(imgs, training=False).numpy()
    labels_np = labels.numpy()
    objectness_mask = labels_np[..., 0] == 1

    true_classes = np.argmax(labels_np[..., 5:], axis=-1)
    pred_classes = np.argmax(preds[..., 5:], axis=-1)

    for b in range(labels_np.shape[0]):
        for r in range(8):
            for c in range(8):
                if objectness_mask[b, r, c]:
                    t = true_classes[b, r, c]
                    p = pred_classes[b, r, c]
                    class_total[t] += 1
                    total += 1
                    if t == p:
                        correct += 1
                        class_correct[t] += 1

print(f'Overall class accuracy on positive cells: {correct/total:.1%}')
print()
class_labels = ['Zero', 'One', 'Two', 'Three', 'Four', 'Five']
for i in range(6):
    if class_total[i] > 0:
        print(f'  {class_labels[i]}: {class_correct[i]/class_total[i]:.1%} ({class_correct[i]}/{class_total[i]})')

In [ ]:
per_class_acc = [class_correct[i]/class_total[i] if class_total[i] > 0 else 0 for i in range(6)]

plt.figure(figsize=(8, 5))
plt.bar(class_labels, per_class_acc, color='steelblue')
plt.title('Per-Class Accuracy on Positive Cells')
plt.xlabel('Digit Class')
plt.ylabel('Accuracy')
plt.ylim(0, 1.1)
for i, acc in enumerate(per_class_acc):
    plt.text(i, acc + 0.02, f'{acc:.0%}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## Visual Detections on Test Images
Draw predicted bounding boxes and digit labels on sample test images (without non-max suppression).

In [ ]:
def decode_predictions(output, img_h, img_w, threshold=0.3):
    boxes = []
    for row in range(8):
        for col in range(8):
            objectness = 1 / (1 + np.exp(-output[row, col, 0]))
            if objectness > threshold:
                x, y, w, h = output[row, col, 1], output[row, col, 2], output[row, col, 3], output[row, col, 4]
                class_id = np.argmax(output[row, col, 5:])
                x1 = int((x - w/2) * img_w)
                y1 = int((y - h/2) * img_h)
                x2 = int((x + w/2) * img_w)
                y2 = int((y + h/2) * img_h)
                boxes.append((x1, y1, x2, y2, objectness, class_id))
    return boxes

sample_imgs = []
sample_preds = []
sample_labels = []

for imgs, labels in test_dataset.take(1):
    preds = model(imgs, training=False).numpy()
    for i in range(min(8, len(imgs))):
        sample_imgs.append(imgs[i].numpy())
        sample_preds.append(preds[i])
        sample_labels.append(labels[i].numpy())

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
class_labels_short = ['0', '1', '2', '3', '4', '5']

for i in range(len(sample_imgs)):
    img = (sample_imgs[i] * 255).astype(np.uint8).copy()
    H, W = img.shape[:2]
    boxes = decode_predictions(sample_preds[i], H, W)
    for x1, y1, x2, y2, obj, cls in boxes:
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, f'{class_labels_short[cls]} {obj:.2f}',
                    (x1, max(y1-5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    axes[i].imshow(img)
    axes[i].axis('off')

plt.suptitle('Predicted Detections on Test Images', fontsize=14)
plt.tight_layout()
plt.show()

## Ground Truth vs Prediction Comparison
Side by side — ground truth boxes (blue) vs predicted boxes (green).

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 20))

for i in range(min(4, len(sample_imgs))):
    img_gt = (sample_imgs[i] * 255).astype(np.uint8).copy()
    img_pred = (sample_imgs[i] * 255).astype(np.uint8).copy()
    H, W = img_gt.shape[:2]
    label = sample_labels[i]

    # Draw ground truth
    for row in range(8):
        for col in range(8):
            if label[row, col, 0] == 1:
                x, y, w, h = label[row, col, 1], label[row, col, 2], label[row, col, 3], label[row, col, 4]
                cls = np.argmax(label[row, col, 5:])
                x1, y1 = int((x - w/2)*W), int((y - h/2)*H)
                x2, y2 = int((x + w/2)*W), int((y + h/2)*H)
                cv2.rectangle(img_gt, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(img_gt, str(cls), (x1, max(y1-5, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # Draw predictions
    boxes = decode_predictions(sample_preds[i], H, W)
    for x1, y1, x2, y2, obj, cls in boxes:
        cv2.rectangle(img_pred, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img_pred, f'{cls} {obj:.2f}', (x1, max(y1-5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    axes[i, 0].imshow(img_gt)
    axes[i, 0].set_title('Ground Truth (blue)', fontsize=10)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(img_pred)
    axes[i, 1].set_title('Prediction (green)', fontsize=10)
    axes[i, 1].axis('off')

plt.suptitle('Ground Truth vs Prediction', fontsize=14)
plt.tight_layout()
plt.show()